# Lab 03 — Count-Based Word Embeddings

**NLP Applications Laboratory | KLEF | 2026**

*From orthogonal one-hots to context-aware vectors — how counting neighbors gave words meaning.*

## Objective

In Lab 02, we discovered that one-hot vectors fail to capture meaning — every word is equidistant from every other word, and `cat` is no closer to `dog` than to `pizza`.

This lab is the first step out of that wall. By the end, you will be able to:

1. Explain the **distributional hypothesis** — the idea that has driven every embedding technique since 1957.
2. Build **Bag-of-Words (BoW)** representations of sentences.
3. Build a **word co-occurrence matrix** — the first vector space where similar words get similar vectors.
4. Compute **TF-IDF** to weight informative words higher than filler words.
5. Compare all these methods against one-hot on the same word pairs, and *measure* how much meaning each method captures.

We will use the **same 12-sentence corpus** from Lab 02 so you can directly compare results.

## What you will learn before we start

This lab moves through **two big ideas**, both from the pre-neural era of NLP (1950s–2000s). Neither uses a neural network — they are all built from **counting** words and their neighbors.

### Stage 1 — The distributional hypothesis

The core insight of the entire lab:

> *"You shall know a word by the company it keeps."* — J.R. Firth, 1957

Two words that appear near the same other words tend to mean similar things. `cat` and `dog` are both often near `pet`, `chase`, `home`. That shared context is what we will capture as a vector.

**Concept**: what a co-occurrence vector is, and why it captures meaning.

### Stage 2 — Count-based representations

Four related methods, all built on counting:

| Method | What it counts |
|---|---|
| **Bag of Words (BoW)** | Word counts per sentence |
| **TF-IDF** | BoW, but rare words weighted higher |
| **Co-occurrence matrix** | Word counts within a small window around each other word |
| **PPMI** (positive pointwise mutual information) | Cleaner version of raw co-occurrence |

At the end, we compare all five methods (including one-hot from Lab 02) side-by-side on the same word pairs. You will *see* the numbers change as each method captures more meaning.

### What you already have from Lab 02

- The 12-sentence corpus
- The tokenization and vocabulary pipeline
- The one-hot baseline that we will beat

We will rebuild these at the top of the notebook so this lab runs standalone.

In [1]:
# Rebuild the pipeline from Lab 02: corpus → tokens → vocab → indices
corpus = [
    "The cat chased the mouse",
    "Dogs love to play in the park",
    "Rabbits hop through the field",
    "The mouse ran from the cat",
    "Pizza is baked in an oven",
    "I ate pasta for dinner",
    "Bread is made from flour",
    "The oven baked fresh bread",
    "Cars drive on the highway",
    "The bus arrives at eight",
    "Trucks carry heavy loads",
    "A car and a truck raced",
]
tokenized = [s.lower().split() for s in corpus]
vocab = sorted({tok for sent in tokenized for tok in sent})
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
print(f"Corpus: {len(corpus)} sentences | Vocabulary: {len(vocab)} words")

Corpus: 12 sentences | Vocabulary: 47 words


## Stage 1 — The distributional hypothesis in action

The idea:

> Every word has a **context** — the words that appear near it in real sentences. If two words tend to appear in similar contexts, they probably mean similar things.

To measure "context", we define a **window size**. A window of `2` means: for each word in a sentence, look at the 2 words to its left and the 2 words to its right.

### Example

Take sentence 1 from our corpus:

`the cat chased the mouse`

With window size 2, the context of each word is:

| Center word | Left context | Right context |
|---|---|---|
| `the` (1st) | — | `cat`, `chased` |
| `cat` | `the` | `chased`, `the` |
| `chased` | `the`, `cat` | `the`, `mouse` |
| `the` (2nd) | `cat`, `chased` | `mouse` |
| `mouse` | `chased`, `the` | — |

Notice that **`cat` sees `chased` in its context**, and so does `mouse`. That's a very small hint that `cat` and `mouse` might be related — they both appear near `chased`.

Now do this counting across the entire corpus, and the hint becomes a signal.

# Build the co-occurrence counts

### Now we do exactly what the concept cell described, but for the entire corpus. For each word in each sentence, count which other words appear within a window of 2.

In [2]:
# For each word, count how often each other word appears within window=2
from collections import defaultdict
window = 2
cooc_counts = defaultdict(lambda: defaultdict(int))
for sent in tokenized:
    for i, center in enumerate(sent):
        for j in range(max(0, i-window), min(len(sent), i+window+1)):
            if i != j:
                cooc_counts[center][sent[j]] += 1
print(f"cooc_counts['cat']    = {dict(cooc_counts['cat'])}")
print(f"cooc_counts['mouse']  = {dict(cooc_counts['mouse'])}")

cooc_counts['cat']    = {'the': 3, 'chased': 1, 'from': 1}
cooc_counts['mouse']  = {'chased': 1, 'the': 2, 'ran': 1, 'from': 1}


Look carefully at the two dictionaries.

cat sees these context words: {the, chased, from, mouse, ran}
mouse sees these context words: {chased, the, cat, ran, from}

They are nearly identical. Both words see the, chased, ran, from around them. cat sees mouse, and mouse sees cat, because they appear in the same two sentences.

This is what the distributional hypothesis predicts: words with similar meanings appear in similar contexts. cat and mouse share context in this corpus because they both appear in animal-chase scenarios. That similarity of context is what we're about to turn into a numeric similarity.

In [3]:
print(f"cooc_counts['pizza']  = {dict(cooc_counts['pizza'])}")
print(f"cooc_counts['bread']  = {dict(cooc_counts['bread'])}")

cooc_counts['pizza']  = {'is': 1, 'baked': 1}
cooc_counts['bread']  = {'is': 1, 'made': 1, 'baked': 1, 'fresh': 1}


In [4]:
# Turn the counts into proper vectors

In [5]:
# Build a V x V co-occurrence matrix where row i is the context vector for word i
import numpy as np
V = len(vocab)
cooc_matrix = np.zeros((V, V), dtype=int)
for center_word, contexts in cooc_counts.items():
    i = word2idx[center_word]
    for context_word, count in contexts.items():
        cooc_matrix[i, word2idx[context_word]] = count
print(f"Shape: {cooc_matrix.shape}")
print(f"Total nonzero entries: {(cooc_matrix > 0).sum()}")

Shape: (47, 47)
Total nonzero entries: 170


In [28]:
cooc_matrix

array([[0, 0, 2, ..., 0, 1, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [2, 0, 0, ..., 0, 1, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [1, 0, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(47, 47))

In [29]:
# The full V x V co-occurrence matrix with word labels on both axes
cooc_df = pd.DataFrame(cooc_matrix, index=vocab, columns=vocab)
cooc_df

,a,an,and,arrives,at,ate,baked,bread,bus,car,...,pizza,play,rabbits,raced,ran,the,through,to,truck,trucks
a,0,0,2,0,0,0,0,0,0,2,...,0,0,0,1,0,0,0,0,1,0
an,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
and,2,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0
arrives,0,0,0,0,1,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
at,0,0,0,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
ate,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
baked,0,1,0,0,0,0,0,1,0,0,...,1,0,0,0,0,1,0,0,0,0
bread,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
bus,0,0,0,1,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
car,2,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
print(f"Row for 'cat' (first 20 elements): {cooc_matrix[word2idx['cat']][:20]}")
print(f"Row for 'mouse' (first 20 elements): {cooc_matrix[word2idx['mouse']][:20]}")

Row for 'cat' (first 20 elements): [0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0]
Row for 'mouse' (first 20 elements): [0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0]


In [30]:
# Focus on cluster-defining words so the semantic structure jumps out
key = ['cat', 'mouse', 'dogs', 'rabbits', 'chased', 'ran',
       'pizza', 'bread', 'oven', 'baked',
       'car', 'cars', 'bus', 'truck', 'trucks',
       'the', 'a', 'is']
present = [w for w in key if w in vocab]
cooc_df.loc[present, present]

,cat,mouse,dogs,rabbits,chased,ran,pizza,bread,oven,baked,car,cars,bus,truck,trucks,the,a,is
cat,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,3,0,0
mouse,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,2,0,0
dogs,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
rabbits,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
chased,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0
ran,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0
pizza,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1
bread,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1
oven,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0
baked,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,1,0,1


In [31]:
# Show all context words for a given word — the word's "distributional footprint"
def show_context(word):
    row = cooc_df.loc[word]
    nonzero = row[row > 0].sort_values(ascending=False)
    print(f"'{word}' saw these words in its context (window=2):")
    print(f"{'Context word':<15} {'Count':>6}")
    print("-" * 25)
    for cw, count in nonzero.items():
        print(f"{cw:<15} {count:>6}")
    print()

show_context('cat')
show_context('mouse')
show_context('pizza')

'cat' saw these words in its context (window=2):
Context word     Count
-------------------------
the                  3
chased               1
from                 1

'mouse' saw these words in its context (window=2):
Context word     Count
-------------------------
the                  2
chased               1
from                 1
ran                  1

'pizza' saw these words in its context (window=2):
Context word     Count
-------------------------
baked                1
is                   1



# Compute similarity between word pairs using co-occurrence vectors

In [7]:
# Same cosine similarity function from Lab 02, now applied to co-occurrence vectors
def cosine_similarity(v1, v2):
    dot = np.dot(v1, v2)
    norm = np.linalg.norm(v1) * np.linalg.norm(v2)
    return dot / norm if norm > 0 else 0.0

pairs = [('cat', 'mouse'), ('cat', 'pizza'), ('pizza', 'bread'), ('cat', 'truck')]
for w1, w2 in pairs:
    v1 = cooc_matrix[word2idx[w1]]
    v2 = cooc_matrix[word2idx[w2]]
    sim = cosine_similarity(v1, v2)
    print(f"sim({w1:>6}, {w2:<6}) = {sim:.4f}")

sim(   cat, mouse ) = 0.9117
sim(   cat, pizza ) = 0.0000
sim( pizza, bread ) = 0.7071
sim(   cat, truck ) = 0.0000


### Stage 1 wrap-up — we broke the wall

Compare with what one-hot gave us in Lab 02:

| Pair | One-hot (Lab 02) | Co-occurrence (this lab) |
|---|---|---|
| `cat`, `mouse` (same cluster) | 0.0000 | **0.9117** |
| `pizza`, `bread` (same cluster) | 0.0000 | **0.7071** |
| `cat`, `pizza` (different clusters) | 0.0000 | 0.0000 |
| `cat`, `truck` (different clusters) | 0.0000 | 0.0000 |

**Same-cluster pairs are now high, different-cluster pairs are still low.** This is exactly what an embedding is supposed to do.

The idea that made this work is deceptively simple: **each word is represented by the words that appear next to it**. Nothing about `cat` changed — we just gave it a richer vector that reflects *what company it keeps* in the corpus.

### One honest caveat

The two zeros (`cat-pizza`, `cat-truck`) look "correct" but are actually **artifacts of a tiny corpus**. In a real corpus, different-cluster words would share function words like `the`, `a`, `is` — so the similarity would be *low but nonzero*. The perfect zeros here happen because our 12 sentences are too short to force such sharing.

### The problems Stage 1 still has

- **Raw counts are noisy.** The word `the` co-occurs with *everything* — its high count carries no information.
- **Sparsity.** The 47×47 matrix is still ~85% zeros. A real vocabulary of 100,000 words would give us a 10-billion-entry matrix.
- **Bag of Words (BoW)** for sentences — the natural cousin of co-occurrence for words — will let us represent *entire sentences* as vectors, opening up document-level similarity.

These are what Stage 2 addresses next.

In [8]:
from nlpa_chat import chat
chat()

## Stage 2 — Sentence-level vectors: Bag of Words

In Stage 1 we asked: *"What are each word's neighbors?"* — and got a **vector per word**.

Now we ask a different question: *"Which words appear in each sentence?"* — and get a **vector per sentence**.

This is called **Bag of Words (BoW)** because we throw away word order and just count occurrences. The sentence `the cat chased the mouse` becomes the same vector as `mouse the chased cat the` — we treat sentences as unordered "bags" of words.

Why this matters:

- A **word vector** (Stage 1) tells us what a word means.
- A **sentence vector** (Stage 2) tells us what a whole document is *about*.
- Both are built by counting — but at different scales.

BoW is the foundation of **information retrieval** — every classical search engine and text classifier from the 1960s through the 2010s was built on BoW or its close cousin TF-IDF.

# Build the Bag of Words matrix

In [9]:
# BoW: each sentence becomes a vector of word counts (indexed by vocabulary)
bow_matrix = np.zeros((len(corpus), V), dtype=int)
for sent_idx, sent in enumerate(tokenized):
    for word in sent:
        bow_matrix[sent_idx, word2idx[word]] += 1
print(f"BoW matrix shape: {bow_matrix.shape}")
print(f"Sentence 1: {corpus[0]}")
print(f"Its vector (first 20 elements): {bow_matrix[0, :20]}")

BoW matrix shape: (12, 47)
Sentence 1: The cat chased the mouse
Its vector (first 20 elements): [0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0]


In [10]:
# See the BoW matrix with word labels — plain Python, no pandas
key_words = ['cat', 'mouse', 'dogs', 'rabbits', 'pizza', 'bread', 'oven', 'pasta',
             'car', 'cars', 'bus', 'truck', 'trucks']
key_idx = [word2idx[w] for w in key_words]
print(f"{'Sentence':<32} | " + " ".join(f"{w[:5]:>5}" for w in key_words))
print("-" * 120)
for i, sent in enumerate(corpus):
    counts = [bow_matrix[i, j] for j in key_idx]
    print(f"S{i+1:>2}: {sent[:26]:<26} | " + " ".join(f"{c:>5}" for c in counts))

Sentence                         |   cat mouse  dogs rabbi pizza bread  oven pasta   car  cars   bus truck truck
------------------------------------------------------------------------------------------------------------------------
S 1: The cat chased the mouse   |     1     1     0     0     0     0     0     0     0     0     0     0     0
S 2: Dogs love to play in the p |     0     0     1     0     0     0     0     0     0     0     0     0     0
S 3: Rabbits hop through the fi |     0     0     0     1     0     0     0     0     0     0     0     0     0
S 4: The mouse ran from the cat |     1     1     0     0     0     0     0     0     0     0     0     0     0
S 5: Pizza is baked in an oven  |     0     0     0     0     1     0     1     0     0     0     0     0     0
S 6: I ate pasta for dinner     |     0     0     0     0     0     0     0     1     0     0     0     0     0
S 7: Bread is made from flour   |     0     0     0     0     0     1     0     0     0     0 

In [11]:
# See the BoW matrix with actual word labels — sentences as rows, vocab as columns
import pandas as pd
bow_df = pd.DataFrame(bow_matrix, columns=vocab)
bow_df.index = [f"S{i+1}: {corpus[i][:30]}" for i in range(len(corpus))]
bow_df.head(12)

,a,an,and,arrives,at,ate,baked,bread,bus,car,...,pizza,play,rabbits,raced,ran,the,through,to,truck,trucks
S1: The cat chased the mouse,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,2,0,0,0,0
S2: Dogs love to play in the park,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,1,0,1,0,0
S3: Rabbits hop through the field,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,1,1,0,0,0
S4: The mouse ran from the cat,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,2,0,0,0,0
S5: Pizza is baked in an oven,0,1,0,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0
S6: I ate pasta for dinner,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
S7: Bread is made from flour,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
S8: The oven baked fresh bread,0,0,0,0,0,0,1,1,0,0,...,0,0,0,0,0,1,0,0,0,0
S9: Cars drive on the highway,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
S10: The bus arrives at eight,0,0,0,1,1,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0


# Sentence similarity using BoW vectors

In [12]:
# Compute cosine similarity between every pair of sentences
n = len(corpus)
sent_sim = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sent_sim[i, j] = cosine_similarity(bow_matrix[i], 
                                           bow_matrix[j])
print(f"Sentence similarity matrix shape: {sent_sim.shape}")
print(f"S1 ↔ S4 (both cat-mouse): {sent_sim[0, 3]:.4f}")
print(f"S1 ↔ S5 (cat vs pizza)  : {sent_sim[0, 4]:.4f}")
print(f"S5 ↔ S8 (both pizza-bread): {sent_sim[4, 7]:.4f}")
print(f"S9 ↔ S12 (both car sentences): {sent_sim[8, 11]:.4f}")

Sentence similarity matrix shape: (12, 12)
S1 ↔ S4 (both cat-mouse): 0.8018
S1 ↔ S5 (cat vs pizza)  : 0.0000
S5 ↔ S8 (both pizza-bread): 0.3651
S9 ↔ S12 (both car sentences): 0.0000


### Same-cluster sentences (S1↔S4, S5↔S8, S9↔S12) get meaningfully higher similarity than cross-cluster (S1↔S5).
#### Even without any semantic understanding, BoW captures thematic overlap through shared vocabulary.
#### The similarities aren't perfect — they're often driven by shared function words like the, a, in rather than the thematic words. That's the known weakness of raw BoW and the reason TF-IDF exists.

In [13]:
# Investigate why S9 and S12 have zero similarity
print(f"S9  ({corpus[8]}): {tokenized[8]}")
print(f"S12 ({corpus[11]}): {tokenized[11]}")
print(f"Words in common: {set(tokenized[8]) & set(tokenized[11])}")

S9  (Cars drive on the highway): ['cars', 'drive', 'on', 'the', 'highway']
S12 (A car and a truck raced): ['a', 'car', 'and', 'a', 'truck', 'raced']
Words in common: set()


### The **S9 ↔ S12 failure** exposes **BoW's** fundamental limitation: it treats every word form as unique. `car` and `cars` are the same concept but different vocabulary entries. BoW cannot see they're related.


# Stage 2 **TF-IDF**, which addresses problem #2 — weighting rare informative words higher and common function words lower.

where `N` is the total number of sentences and `df(w)` is how many sentences contain `w`.

Intuition:

- Words appearing in **every sentence** (like `the`) → IDF is near 0 → TF-IDF near 0. Suppressed.
- Words appearing in **one sentence** (like `pizza`) → IDF is high → TF-IDF is high. Amplified.

TF-IDF was **the standard** for information retrieval and text classification from the 1970s through the 2010s. Every classical search engine — pre-BERT Google, early Bing — ranked results using variants of TF-IDF.

# Compute TF-IDF using scikit-learn

In [17]:
# Use scikit-learn's TfidfVectorizer — the industry standard
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vec = TfidfVectorizer(lowercase=True, token_pattern=r'\S+')
tfidf_matrix = tfidf_vec.fit_transform(corpus).toarray()
tfidf_vocab = tfidf_vec.get_feature_names_out()
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Vocabulary size    : {len(tfidf_vocab)}")

TF-IDF matrix shape: (12, 47)
Vocabulary size    : 47


In [16]:
!pip install scikit-learn

  Using cached scikit_learn-1.9.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.9.0-cp311-cp311-win_amd64.whl (8.3 MB)
Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl (36.6 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ---------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.6.1 requires ml_dtypes>=0.5.0, which is not installed.
jax 0.6.1 requires opt_einsum, which is not installed.
jaxlib 0.6.1 requires ml_dtypes>=0.2.0, which is not installed.


In [18]:
# Show TF-IDF values for sentence 5 (the pizza sentence) — see which words get high weight
sent_idx = 4  # 0-indexed, so this is S5 "Pizza is baked in an oven"
scores = list(zip(tfidf_vocab, tfidf_matrix[sent_idx]))
scores.sort(key=lambda x: -x[1])
print(f"S5: {corpus[sent_idx]}\n")
print(f"{'Word':<12} {'TF-IDF':>8}")
for word, score in scores[:8]:
    if score > 0:
        print(f"{word:<12} {score:>8.4f}")

S5: Pizza is baked in an oven

Word           TF-IDF
an             0.4495
pizza          0.4495
baked          0.3860
in             0.3860
is             0.3860
oven           0.3860


Notice an is tied with pizza at the top. In our theoretical explanation of TF-IDF, we said pizza (rare word) should be much higher than an (common word). But your data shows they're identical.

Why? Look at your corpus — an only appears in sentence 5 ("Pizza is baked in an oven"). Nowhere else. So in this tiny 12-sentence corpus, an is just as rare as pizza. Both appear in exactly one document, so both get identical IDF scores.

This is a great teaching moment: TF-IDF's power scales with corpus size. In a 10,000-document corpus, an would appear in maybe 8,000 documents and pizza in 20 — the ranking would be dramatic. In 12 documents, "rare" words include function words that happen not to appear elsewhere.

# The full TF-IDF matrix as a labeled table

In [19]:
# The whole TF-IDF matrix with sentence labels and vocabulary as columns
tfidf_df = pd.DataFrame(tfidf_matrix, columns=tfidf_vocab)
tfidf_df.index = [f"S{i+1}" for i in range(len(corpus))]
tfidf_df.round(2).head(12)

,a,an,and,arrives,at,ate,baked,bread,bus,car,...,pizza,play,rabbits,raced,ran,the,through,to,truck,trucks
S1,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.55,0.00,0.00,0.00,0.0
S2,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.41,0.00,0.00,0.00,0.21,0.00,0.41,0.00,0.0
S3,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.48,0.00,0.00,0.25,0.48,0.00,0.00,0.0
S4,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.48,0.50,0.00,0.00,0.00,0.0
S5,0.00,0.45,0.00,0.00,0.00,0.00,0.39,0.00,0.00,0.00,...,0.45,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0
S6,0.00,0.00,0.00,0.00,0.00,0.45,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0
S7,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.42,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0
S8,0.00,0.00,0.00,0.00,0.00,0.00,0.46,0.46,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.28,0.00,0.00,0.00,0.0
S9,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.25,0.00,0.00,0.00,0.0
S10,0.00,0.00,0.00,0.48,0.48,0.00,0.00,0.00,0.48,0.00,...,0.00,0.00,0.00,0.00,0.00,0.25,0.00,0.00,0.00,0.0


# Focused view on the "cluster words" only

In [20]:
# Zoom in on cluster words to see TF-IDF's semantic focus
key_words = ['cat', 'mouse', 'dogs', 'rabbits', 'pizza', 'bread', 'oven', 'pasta',
             'car', 'cars', 'bus', 'truck', 'trucks', 'the', 'a', 'is']
present = [w for w in key_words if w in tfidf_vocab]
tfidf_df[present].round(3)

,cat,mouse,dogs,rabbits,pizza,bread,oven,pasta,car,cars,bus,truck,trucks,the,a,is
S1,0.456,0.456,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.549,0.000,0.000
S2,0.000,0.000,0.408,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.211,0.000,0.000
S3,0.000,0.000,0.000,0.484,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.250,0.000,0.000
S4,0.415,0.415,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.500,0.000,0.000
S5,0.000,0.000,0.000,0.000,0.449,0.000,0.386,0.000,0.000,0.000,0.000,0.000,0.0,0.000,0.000,0.386
S6,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.447,0.000,0.000,0.000,0.000,0.0,0.000,0.000,0.000
S7,0.000,0.000,0.000,0.000,0.000,0.418,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.000,0.000,0.418
S8,0.000,0.000,0.000,0.000,0.000,0.460,0.460,0.000,0.000,0.000,0.000,0.000,0.0,0.277,0.000,0.000
S9,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.484,0.000,0.000,0.0,0.250,0.000,0.000
S10,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.484,0.000,0.0,0.250,0.000,0.000


# A per-word view: which sentences care about "pizza"?

In [21]:
# Column view: which sentences give the word "pizza" a nonzero score?
word = 'pizza'
col = tfidf_df[word].round(3)
print(f"TF-IDF values for '{word}' across all sentences:\n")
for sent_label, score in col.items():
    marker = "  <-- nonzero here" if score > 0 else ""
    print(f"  {sent_label} ({corpus[int(sent_label[1:])-1][:35]}): {score:.4f}{marker}")

TF-IDF values for 'pizza' across all sentences:

  S1 (The cat chased the mouse): 0.0000
  S2 (Dogs love to play in the park): 0.0000
  S3 (Rabbits hop through the field): 0.0000
  S4 (The mouse ran from the cat): 0.0000
  S5 (Pizza is baked in an oven): 0.4490  <-- nonzero here
  S6 (I ate pasta for dinner): 0.0000
  S7 (Bread is made from flour): 0.0000
  S8 (The oven baked fresh bread): 0.0000
  S9 (Cars drive on the highway): 0.0000
  S10 (The bus arrives at eight): 0.0000
  S11 (Trucks carry heavy loads): 0.0000
  S12 (A car and a truck raced): 0.0000


## The word **pizza** is a single-sentence signal. It appears in only S5, so its TF-IDF is nonzero only there. Any classifier or search engine using this column can immediately know "queries mentioning pizza should match S5, nothing else."

# Compare with a common word like **the**:

In [22]:
word = 'the'
col = tfidf_df[word].round(3)
print(f"TF-IDF values for '{word}' across all sentences:\n")
for sent_label, score in col.items():
    print(f"  {sent_label}: {score:.4f}")

TF-IDF values for 'the' across all sentences:

  S1: 0.5490
  S2: 0.2110
  S3: 0.2500
  S4: 0.5000
  S5: 0.0000
  S6: 0.0000
  S7: 0.0000
  S8: 0.2770
  S9: 0.2500
  S10: 0.2500
  S11: 0.0000
  S12: 0.0000


### Compare with pizza. **pizza** is a strong signal in one place. the is a weak signal in many places. To a machine, **pizza** is much more useful for distinguishing sentences than **the** is.

# Sentence Similarity using TF-IDF vectors, compared with BoW

In [23]:
# Same cosine similarity function, now applied to TF-IDF vectors
tfidf_sim = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        tfidf_sim[i, j] = cosine_similarity(tfidf_matrix[i], tfidf_matrix[j])
print(f"{'Pair':<40} {'BoW':>8} {'TF-IDF':>8}")
print("-" * 60)
pairs = [(0, 3, 'S1 ↔ S4 (both cat-mouse)'),
         (4, 7, 'S5 ↔ S8 (both pizza-bread)'),
         (8, 11, 'S9 ↔ S12 (both car sentences)'),
         (0, 4, 'S1 ↔ S5 (cat vs pizza)'),
         (4, 8, 'S5 ↔ S9 (pizza vs car)')]
for i, j, label in pairs:
    print(f"{label:<40} {sent_sim[i, j]:>8.4f} {tfidf_sim[i, j]:>8.4f}")

Pair                                          BoW   TF-IDF
------------------------------------------------------------
S1 ↔ S4 (both cat-mouse)                   0.8018   0.6532
S5 ↔ S8 (both pizza-bread)                 0.3651   0.3554
S9 ↔ S12 (both car sentences)              0.0000   0.0000
S1 ↔ S5 (cat vs pizza)                     0.0000   0.0000
S5 ↔ S9 (pizza vs car)                     0.0000   0.0000


### BoW vs TF-IDF — reading the numbers

Same 5 pairs, two vectorizers:

| Pair | BoW | TF-IDF | What changed |
|---|---|---|---|
| S1 ↔ S4 (cat-mouse) | 0.8018 | **0.6532** | Down by 19% — `the` was inflating BoW |
| S5 ↔ S8 (pizza-bread) | 0.3651 | 0.3554 | Nearly unchanged — mostly content words shared |
| S9 ↔ S12 (car sentences) | 0.0000 | 0.0000 | Same problem: `car` ≠ `cars` to both |
| S1 ↔ S5 (cat vs pizza) | 0.0000 | 0.0000 | Correctly identifies unrelated |
| S5 ↔ S9 (pizza vs car) | 0.0000 | 0.0000 | Correctly identifies unrelated |

**Counterintuitive at first**: TF-IDF gave *lower* numbers for same-cluster pairs. Isn't lower worse?

**No.** BoW's 0.8018 was inflated by counting `the` (which appears in both sentences and everywhere else in the corpus). TF-IDF suppresses `the`'s contribution and leaves only what genuinely distinguishes the sentences. The 0.6532 is a **truer** measure of similarity — smaller in magnitude but more meaningful.

Think of BoW as loud and cluttered; TF-IDF as quieter and cleaner.

### The problem that persists

Both methods **still return 0.0000 for S9 ↔ S12** — even though both sentences are about vehicles. Why?

- S9 says `cars` (plural).
- S12 says `car` (singular).
- They share **zero** vocabulary entries.

TF-IDF is a re-weighting of BoW — it can't invent similarity where there's no vocabulary overlap. This is the **morphology blind spot**, and it's a wall that only Lab 04's dense embeddings will fully break.

**Preview of Lab 04**: with modern Word2Vec / GloVe / BERT embeddings, `car` and `cars` will end up in almost the same place in vector space. The 0.0000 will become something like 0.85. Same words, richer representation.

In [24]:
from nlpa_chat import chat
chat()

@cell 16 — Why did TF-IDF give a LOWER similarity for S1 ↔ S4 than BoW? Shouldn't a better method give higher numbers?
@cell 16 — Both BoW and TF-IDF say S9 and S12 have zero similarity. What one change to the corpus would fix this?
If I built a search engine using TF-IDF and a user searched for automobile, would it find Cars drive on the highway? Why or why not?

# Full Comparison

In [25]:
# Grand comparison: all methods so far on the same word pairs
# For word-level pairs, we compare the vectors from each method
onehot_word_matrix = np.eye(V, dtype=int)  # rebuild one-hot from Lab 02

def sim(matrix, w1, w2):
    return cosine_similarity(matrix[word2idx[w1]], matrix[word2idx[w2]])

word_pairs = [('cat', 'mouse'), ('pizza', 'bread'), ('cars', 'car'),
              ('cat', 'pizza'), ('cat', 'truck')]
print(f"{'Word pair':<22} {'One-hot':>10} {'Co-occur':>10}")
print("-" * 45)
for w1, w2 in word_pairs:
    if w1 in word2idx and w2 in word2idx:
        print(f"({w1:>6}, {w2:<7})       "
              f"{sim(onehot_word_matrix, w1, w2):>10.4f} "
              f"{sim(cooc_matrix, w1, w2):>10.4f}")
    else:
        missing = w1 if w1 not in word2idx else w2
        print(f"({w1:>6}, {w2:<7})       {'N/A':>10} {'N/A':>10}   (missing: {missing})")

Word pair                 One-hot   Co-occur
---------------------------------------------
(   cat, mouse  )           0.0000     0.9117
( pizza, bread  )           0.0000     0.7071
(  cars, car    )           0.0000     0.0000
(   cat, pizza  )           0.0000     0.0000
(   cat, truck  )           0.0000     0.0000


# Display the co-occurrence matrix as a labeled table

In [26]:
# The full co-occurrence matrix with word labels on both axes
cooc_df = pd.DataFrame(cooc_matrix, index=vocab, columns=vocab)
cooc_df.head(20)

,a,an,and,arrives,at,ate,baked,bread,bus,car,...,pizza,play,rabbits,raced,ran,the,through,to,truck,trucks
a,0,0,2,0,0,0,0,0,0,2,...,0,0,0,1,0,0,0,0,1,0
an,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
and,2,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0
arrives,0,0,0,0,1,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
at,0,0,0,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
ate,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
baked,0,1,0,0,0,0,0,1,0,0,...,1,0,0,0,0,1,0,0,0,0
bread,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
bus,0,0,0,1,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
car,2,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [27]:
# Focus on cluster words so the pattern becomes visible
key = ['cat', 'mouse', 'dogs', 'rabbits', 'chased', 'ran',
       'pizza', 'bread', 'oven', 'baked',
       'car', 'cars', 'bus', 'truck', 'trucks',
       'the', 'a', 'is']
present = [w for w in key if w in vocab]
cooc_df.loc[present, present]

,cat,mouse,dogs,rabbits,chased,ran,pizza,bread,oven,baked,car,cars,bus,truck,trucks,the,a,is
cat,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,3,0,0
mouse,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,2,0,0
dogs,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
rabbits,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
chased,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0
ran,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0
pizza,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1
bread,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1
oven,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0
baked,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,1,0,1


---

## 📝 Post-Lab Assignment — Lab 03

Complete these questions **at home** and submit as `03_count_based_embeddings_YOURNAME.ipynb` by the next lab session.

You may use the `chat()` widget freely to explore ideas, but **your final answers must be your own words and your own code**. Copying Qwen's output verbatim will be treated as academic dishonesty.

### Part A — Observation (5 marks)

Run the code from Lab 03 and answer using the outputs.

**A1.** Look at the co-occurrence matrix for `cat`. Which 5 words appear in its context, and what are their counts? Present as a table.

**A2.** Look at the co-occurrence matrix for `pizza`. Which words appear in its context? Present as a table.

**A3.** Explain in one sentence why `cat` and `pizza` had cosine similarity of `0.0000` in the co-occurrence method — despite the fact that in real-world text, "eating pizza" and "cats eating things" are both common concepts.

### Part B — Modification (10 marks)

Modify the corpus and observe what changes.

**B1.** Add these three sentences to `corpus`:

```python
"Cats love eating fish",
"The dog ate pizza yesterday",
"A rabbit and a cat sat together",
```

Rebuild `tokenized`, `vocab`, `word2idx`, and `cooc_matrix` from scratch. Report:
- New vocabulary size
- New similarity between `cat` and `mouse`
- New similarity between `cat` and `pizza`
- New similarity between `cat` and `dog`

**B2.** For each of the four numbers above, explain in 1-2 sentences whether it went up, down, or stayed the same compared to Lab 03's original corpus — and *why*.

**B3.** In your modified corpus, which single word has the highest co-occurrence count with `cat`? Print the count. Explain what this means about your corpus, not just the numerics.

### Part C — Design (10 marks)

Propose and test improvements.

**C1.** In the original Lab 03 corpus, `cars` and `car` had similarity `0.0000`. Design a **minimal fix** — add or modify sentences so that the similarity becomes greater than `0.3`. You may add up to 3 new sentences. Report your final similarity and justify your design choice in 2-3 sentences.

**C2.** The window size in our co-occurrence was 2. Repeat the entire pipeline with **window size 5** and window size 1. Report the similarity between `cat` and `mouse` for all three window sizes (1, 2, 5). Which window size gives the highest similarity, and can you explain why?

**C3.** TF-IDF gave *lower* numbers than BoW for same-cluster pairs. A student in your class argues: *"Lower is worse. Therefore BoW is better than TF-IDF."* Write a 2-3 sentence rebuttal explaining why the student is wrong.

### Part D — Analysis and Reflection (10 marks)

Answer thoughtfully in your own words. Each question expects 3-5 sentences.

**D1.** Suppose you are building a search engine for Telugu Wikipedia articles. Which of the methods from Lab 03 (BoW, TF-IDF, co-occurrence, or one-hot) would you use? Would any of them fail on Telugu text? Discuss.

**D2.** The distributional hypothesis says *"you shall know a word by the company it keeps."* Our results validated this on same-cluster pairs like `cat`-`mouse`. But name **one type of word or one type of relationship** where the distributional hypothesis would fail — where two words with completely different meanings might still appear in similar contexts.

**D3.** All four methods in this lab have vectors of dimensionality V (vocabulary size). For a real-world corpus with V = 100,000 words, storing the full co-occurrence matrix would require 10 billion numbers. Propose two different strategies for handling this problem. (You do not need to implement them — just describe the ideas.)

**D4.** Look at your `pizza` context vector from question A2. If we wanted to build a word2vec-style model (Lab 04) that would put `pizza` and `pasta` close in vector space, what would need to be *different* about how we processed this information? Compared to just counting co-occurrences, what would a "learned" model do?

### Part E — Bonus (5 marks, optional)

**E1.** Extend the corpus to 25+ sentences of your own design. Ensure all three clusters (animals, food, vehicles) are represented, plus add a **fourth cluster of your choice** (e.g., musical instruments, weather, sports). Compute the full similarity matrix and identify:
- The two most similar words across the whole vocabulary
- The two most dissimilar words (excluding zero similarities)
- Whether your fourth cluster's words cluster together as intended

Submit your extended corpus, similarity computations, and 2-3 sentences of analysis.

### Total: 40 marks (+5 bonus)

### Deliverables

- Notebook file: `03_count_based_embeddings_YOURNAME.ipynb`
- All cells must be run with visible outputs saved
- Text answers as markdown cells within the notebook

### Deadline

Beginning of next lab session. Late submissions lose 10% per day.